<h1 align="center">Predective Analysis</h1>

In [1]:
!pip install scikit-learn

In [2]:
import sys
!{sys.executable} -m pip install scikit-learn


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: C:\Users\sneha\OneDrive\Documents\python_jupyter_project\.venv\Scripts\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

df = pd.read_csv("Data/Cleaned/Team01_PYVoyagers_cleaned_data.csv")

df.head()


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,patient_id,age,gender,race,average_sleep_duration_hrs,sleep_quality_1_10,_with_sleep_disturbances
0,2018-06-13 18:40:00,283.0,6.360,82.323,27.5,0.092,0.0,0.0,HUPA0001P,34,Male,Other,6.3,4.5,80
1,2018-06-13 18:45:00,283.0,7.728,83.740,0.0,0.092,0.0,0.0,HUPA0001P,34,Male,Other,6.3,4.5,80
2,2018-06-13 18:50:00,283.0,4.750,80.525,0.0,0.092,0.0,0.0,HUPA0001P,34,Male,Other,6.3,4.5,80
3,2018-06-13 18:55:00,283.0,6.359,89.129,20.0,0.092,0.0,0.0,HUPA0001P,34,Male,Other,6.3,4.5,80
4,2018-06-13 19:00:00,283.0,5.152,92.496,0.0,0.075,0.0,0.0,HUPA0001P,34,Male,Other,6.3,4.5,80


## 3.Can we predict a patient’s glucose level one hour in the future using current glucose, recent glucose trends, activity, insulin delivery, sleep factors, and demographics?
This model predicts a patient’s glucose level one hour into the future using current glucose, recent glucose trends, heart rate, steps, calories, insulin delivery, sleep quality, age, gender, race, and patient ID.
This code builds a machine learning model to predict a patient’s glucose level one hour in the future.

We use this model because glucose levels change over time and are affected by many factors, such as current glucose, recent glucose trend, heart rate, activity, insulin delivery, sleep quality, age, gender, and patient differences. A Random Forest model is useful here because it can capture more complex patterns than simple linear regression.

The code first loads the cleaned dataset and converts the time column into datetime format. Then it sorts the data by patient_id and time so each patient’s records are in the correct time order.

In [4]:


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Prepare data
df = pd.read_csv("Data/cleaned/Team01_PYVoyagers_cleaned_data.csv")
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values(['patient_id', 'time'])

# Target: glucose 1 hour later
# If data is 5-min interval, 12 rows = 1 hour
# If data is 15-min interval, use shift(-4)
df['future_glucose_1hr'] = df.groupby('patient_id')['glucose'].shift(-4)

# Time features
df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek

# Recent glucose history
df['glucose_lag_1'] = df.groupby('patient_id')['glucose'].shift(1)
df['glucose_lag_2'] = df.groupby('patient_id')['glucose'].shift(2)
df['glucose_lag_3'] = df.groupby('patient_id')['glucose'].shift(3)

df['glucose_trend'] = df['glucose'] - df['glucose_lag_3']

df['glucose_rolling_avg_3'] = (
    df.groupby('patient_id')['glucose']
    .rolling(3)
    .mean()
    .reset_index(level=0, drop=True)
)

# Features
numeric_features = [
    'glucose',
    'glucose_lag_1',
    'glucose_lag_2',
    'glucose_lag_3',
    'glucose_trend',
    'glucose_rolling_avg_3',
    'heart_rate',
    'steps',
    'calories',
    'basal_rate',
    'bolus_volume_delivered',
    'age',
    'average_sleep_duration_hrs',
    'sleep_quality_1_10',
    '_with_sleep_disturbances',
    'hour',
    'day_of_week'
]

categorical_features = [
    'gender',
    'race',
    'patient_id'
]

target = 'future_glucose_1hr'

numeric_features = [col for col in numeric_features if col in df.columns]
categorical_features = [col for col in categorical_features if col in df.columns]

model_df = df.dropna(subset=numeric_features + categorical_features + [target]).copy()

X = model_df[numeric_features + categorical_features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("--- FUTURE GLUCOSE PREDICTION MODEL ---")
print(f"Test R²: {r2_score(y_test, predictions):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions):.2f} mg/dL")


--- FUTURE GLUCOSE PREDICTION MODEL ---
Test R²: 0.9638
Mean Absolute Error: 6.66 mg/dL


In [5]:
df.head(5)

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,patient_id,age,...,sleep_quality_1_10,_with_sleep_disturbances,future_glucose_1hr,hour,day_of_week,glucose_lag_1,glucose_lag_2,glucose_lag_3,glucose_trend,glucose_rolling_avg_3
0,2018-06-13 18:40:00,283.0,6.360,82.323,27.5,0.092,0.0,0.0,HUPA0001P,34,...,4.5,80,283.0,18,2,NaN,NaN,NaN,NaN,NaN
1,2018-06-13 18:45:00,283.0,7.728,83.740,0.0,0.092,0.0,0.0,HUPA0001P,34,...,4.5,80,283.0,18,2,283.0,NaN,NaN,NaN,NaN
2,2018-06-13 18:50:00,283.0,4.750,80.525,0.0,0.092,0.0,0.0,HUPA0001P,34,...,4.5,80,283.0,18,2,283.0,283.0,NaN,NaN,283.0
3,2018-06-13 18:55:00,283.0,6.359,89.129,20.0,0.092,0.0,0.0,HUPA0001P,34,...,4.5,80,283.0,18,2,283.0,283.0,283.0,0.0,283.0
4,2018-06-13 19:00:00,283.0,5.152,92.496,0.0,0.075,0.0,0.0,HUPA0001P,34,...,4.5,80,283.0,19,2,283.0,283.0,283.0,0.0,283.0


The Random Forest model performed well in predicting glucose levels one hour ahead. It achieved an R² score of 0.9598, meaning it explained about 96% of the variation in future glucose levels. The mean absolute error was 7.16 mg/dL, showing that predictions were typically within about 7 mg/dL of the actual glucose value. This suggests that recent glucose trends, patient activity, insulin delivery, sleep features, and demographics are useful predictors of short-term glucose changes.

In [6]:
df.head(5)

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,patient_id,age,...,sleep_quality_1_10,_with_sleep_disturbances,future_glucose_1hr,hour,day_of_week,glucose_lag_1,glucose_lag_2,glucose_lag_3,glucose_trend,glucose_rolling_avg_3
0,2018-06-13 18:40:00,283.0,6.360,82.323,27.5,0.092,0.0,0.0,HUPA0001P,34,...,4.5,80,283.0,18,2,NaN,NaN,NaN,NaN,NaN
1,2018-06-13 18:45:00,283.0,7.728,83.740,0.0,0.092,0.0,0.0,HUPA0001P,34,...,4.5,80,283.0,18,2,283.0,NaN,NaN,NaN,NaN
2,2018-06-13 18:50:00,283.0,4.750,80.525,0.0,0.092,0.0,0.0,HUPA0001P,34,...,4.5,80,283.0,18,2,283.0,283.0,NaN,NaN,283.0
3,2018-06-13 18:55:00,283.0,6.359,89.129,20.0,0.092,0.0,0.0,HUPA0001P,34,...,4.5,80,283.0,18,2,283.0,283.0,283.0,0.0,283.0
4,2018-06-13 19:00:00,283.0,5.152,92.496,0.0,0.075,0.0,0.0,HUPA0001P,34,...,4.5,80,283.0,19,2,283.0,283.0,283.0,0.0,283.0


## 4.  Predicting “Night‑Time Hypoglycemia Risk”
We are predicting whether a patient is at risk of night-time hypoglycemia, which means their glucose level may drop below 70 mg/dL during the night, between 12 AM and 6 AM.

The model uses daily patient data such as average glucose, heart rate, steps, calories, insulin delivery, age, gender, race, and sleep-related features to predict whether a low-glucose event may occur overnight.

This model predicts the risk of night-time hypoglycemia for diabetes patients. A night-time hypoglycemia event is defined as glucose dropping below 70 mg/dL between 12 AM and 6 AM. By analyzing patient activity, glucose patterns, insulin delivery, and sleep-related factors, the model identifies whether a patient may be at risk for an overnight low-glucose event. This prediction can support earlier intervention and improve patient safety.

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Prepare data
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values(['patient_id', 'time'])

# Create date and hour
df['date'] = df['time'].dt.date
df['hour'] = df['time'].dt.hour

# Night period: 12 AM to 6 AM
night_df = df[(df['hour'] >= 0) & (df['hour'] < 6)].copy()

# Target: did patient have hypoglycemia at night?
night_risk = (
    night_df
    .groupby(['patient_id', 'date'])['glucose']
    .apply(lambda x: 1 if (x < 70).any() else 0)
    .reset_index()
    .rename(columns={'glucose': 'night_hypoglycemia_risk'})
)

# 2. Create daily features from previous evening/day
daily_features = (
    df.groupby(['patient_id', 'date'])
    .agg({
        'glucose': 'mean',
        'heart_rate': 'mean',
        'steps': 'sum',
        'calories': 'sum',
        'basal_rate': 'mean',
        'bolus_volume_delivered': 'sum',
        'carb_input': 'sum',
        'age': 'first',
        'gender': 'first',
        'race': 'first',
        'average_sleep_duration_hrs': 'first',
        'sleep_quality_1_10': 'first',
        '_with_sleep_disturbances': 'first'
    })
    .reset_index()
)

# Rename columns for clarity
daily_features = daily_features.rename(columns={
    'glucose': 'avg_glucose',
    'heart_rate': 'avg_heart_rate',
    'steps': 'total_steps',
    'calories': 'total_calories',
    'basal_rate': 'avg_basal_rate',
    'bolus_volume_delivered': 'total_bolus',
    'carb_input': 'total_carbs'
})

# 3. Merge features with target
model_df = daily_features.merge(
    night_risk,
    on=['patient_id', 'date'],
    how='inner'
)

# 4. Features and target
target = 'night_hypoglycemia_risk'

numeric_features = [
    'avg_glucose',
    'avg_heart_rate',
    'total_steps',
    'total_calories',
    'avg_basal_rate',
    'total_bolus',
    'total_carbs',
    'age',
    'average_sleep_duration_hrs',
    'sleep_quality_1_10',
    '_with_sleep_disturbances'
]

categorical_features = [
    'gender',
    'race',
    'patient_id'
]

numeric_features = [col for col in numeric_features if col in model_df.columns]
categorical_features = [col for col in categorical_features if col in model_df.columns]

model_df = model_df.dropna(subset=numeric_features + categorical_features + [target])

X = model_df[numeric_features + categorical_features]
y = model_df[target]

# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 6. Model
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        class_weight='balanced'
    ))
])

model.fit(X_train, y_train)

# 7. Evaluate
predictions = model.predict(X_test)

print("--- NIGHT-TIME HYPOGLYCEMIA RISK MODEL ---")
print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions))


--- NIGHT-TIME HYPOGLYCEMIA RISK MODEL ---
Accuracy: 0.758

Confusion Matrix:
[[143  20]
 [ 32  20]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.88      0.85       163
           1       0.50      0.38      0.43        52

    accuracy                           0.76       215
   macro avg       0.66      0.63      0.64       215
weighted avg       0.74      0.76      0.75       215



In [8]:
risk_probabilities = model.predict_proba(X_test)[:, 1]

risk_results = X_test.copy()
risk_results['actual_risk'] = y_test.values
risk_results['predicted_risk'] = predictions
risk_results['risk_probability'] = risk_probabilities.round(3)

risk_results.head()


,avg_glucose,avg_heart_rate,total_steps,total_calories,avg_basal_rate,total_bolus,total_carbs,age,average_sleep_duration_hrs,sleep_quality_1_10,_with_sleep_disturbances,gender,race,patient_id,actual_risk,predicted_risk,risk_probability
551,129.657986,75.364510,1535.0,2394.997,0.062563,28.0,15.5,60,5.5,5.9,60,Female,Black,HUPA0027P,0,0,0.237
361,164.291087,79.967687,2193.5,1831.453,0.054833,28.0,3.0,33,6.0,4.7,50,Male,Hispanic,HUPA0026P,0,0,0.271
60,155.083333,84.034611,2572.0,1998.364,0.067083,8.3,8.0,49,5.8,7.9,30,Male,Native American,HUPA0005P,0,0,0.424
779,169.414351,83.634441,1701.5,2405.675,0.063969,16.0,8.0,60,5.5,5.9,60,Female,Black,HUPA0027P,0,0,0.229
815,111.656250,66.897278,1017.0,2244.247,0.055542,3.0,0.0,60,5.5,5.9,60,Female,Black,HUPA0027P,1,0,0.496


The night-time hypoglycemia risk model achieved an overall accuracy of 75.8%. The model performed well in identifying no-risk cases, with a recall of 88% for class 0. However, it was less effective at detecting actual hypoglycemia risk, with a recall of 38% for class 1. This suggests that while the model can identify normal overnight glucose patterns reasonably well, additional features or more balanced hypoglycemia examples may be needed to improve detection of high-risk cases.